In [0]:
%pip install nemosis

In [0]:
# Import necessary libraries
import os
from nemosis import dynamic_data_compiler
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, IntegerType

In [0]:
nsw_dispatch_5min = spark.table("energy_endava_193.default.nsw_dispatch_5min")
nsw_dictionary = spark.table("energy_endava_193.default.nsw_dictionary")
nsw_dictionary_second = spark.table("energy_endava_193.default.nsw_dictionary_second")
nsw_demand = spark.table("energy_endava_193.default.nsw_demand")
nsw_scada = spark.table("energy_endava_193.default.nsw_scada")
# Read "PU and Scheduled Loads" sheet from Excel file using pandas, then convert to Spark DataFrame
nsw_registration_pd = pd.read_excel("/Workspace/Users/quangthanhdong04.au@gmail.com/energy-endava-193/src/energy_endava_193_etl/transformations/dataset/NEM Registration and Exemption List.xlsx", sheet_name="PU and Scheduled Loads")
# Convert all object columns to string to handle mixed types
for col in nsw_registration_pd.select_dtypes(include=['object']).columns:
    nsw_registration_pd[col] = nsw_registration_pd[col].astype(str)
nsw_registration = spark.createDataFrame(nsw_registration_pd)

# Read co2eii_available_generators.csv using pandas
# The file structure: Line 0 is metadata, Line 1 has column headers (starting with 'I'), Line 2+ are data rows (starting with 'D')
raw_generators_pd = pd.read_csv("/Workspace/Users/quangthanhdong04.au@gmail.com/energy-endava-193/src/energy_endava_193_etl/transformations/dataset/co2eii_available_generators.csv", header=None, skiprows=1)
# Use first row (line 1 of file) as header, skipping the first column which is the row type indicator
header = raw_generators_pd.iloc[0, 1:].tolist()
data_generators_pd = raw_generators_pd.iloc[1:, 1:].copy()  # Skip first column (row type) and first row (header)
data_generators_pd.columns = header
nsw_generators = spark.createDataFrame(data_generators_pd)

In [0]:
display(nsw_dictionary_second)

In [0]:
# Mapping qualified generators with their respective "Fuel Source - Primary"
# Using 2 different tables: nsw_registration and nsw_generators
# The generators whose DUIDs are not present in these tables are individually researched and added into the nsw_dictionary table.
from pyspark.sql.functions import trim, upper, when, col, isnull, lit, row_number
from pyspark.sql.window import Window

# Prepare nsw_registration mapping DataFrame with standardized DUID and relevant columns
nsw_reg_fuel = nsw_registration.select(
    trim(upper(col("DUID"))).alias("DUID"),
    col("Fuel Source - Primary"),
    col("Fuel Source - Descriptor")
).distinct()

# Standardize DUID in nsw_dictionary for join
nsw_dict_std = nsw_dictionary.withColumn("DUID", trim(upper(col("DUID"))))

# Left join nsw_dictionary with nsw_registration fuel info
nsw_dictionary_mapped = nsw_dict_std.join(
    nsw_reg_fuel,
    on="DUID",
    how="left"
)

# Standardize DUID in nsw_generators for join
nsw_generators_std = nsw_generators.withColumn("DUID", trim(upper(col("DUID"))))

# Join nsw_dictionary_mapped with nsw_generators to get CO2E_ENERGY_SOURCE
nsw_dictionary_mapped = nsw_dictionary_mapped.join(
    nsw_generators_std.select("DUID", "CO2E_ENERGY_SOURCE"),
    on="DUID",
    how="left"
)

# Update "Fuel Source - Primary" for DUIDs starting with "TUMUT" or "TUMT"
nsw_dictionary_mapped = nsw_dictionary_mapped.withColumn(
    "Fuel Source - Primary",
    when(
        (col("DUID").startswith("TUMUT")) | (col("DUID").startswith("TUMT")) | (col("DUID") == "KIDSPHG2"),
        "Hydro"
    ).otherwise(col("Fuel Source - Primary"))
)

# Map CO2E_ENERGY_SOURCE to "Fuel Source - Primary" only for NA values
nsw_dictionary_mapped = nsw_dictionary_mapped.withColumn(
    "Fuel Source - Primary",
    when(
        isnull(col("Fuel Source - Primary")),
        col("CO2E_ENERGY_SOURCE")
    ).otherwise(col("Fuel Source - Primary"))
)

# Map the generators without information in either tables with Fuel Source - Primary == "Battery Storage" and Fuel Source - Descriptor == "Grid"
nsw_dictionary_mapped = nsw_dictionary_mapped.withColumn(
    "Fuel Source - Primary",
    when(
        (col("DISPATCHTYPE") != "LOAD") &
        (col("STARTTYPE") != "NOT DISPATCHED") &
        (col("Fuel Source - Primary").isNull()) &
        (~col("DUID").isin(["DG_NSW1", "DG_VIC1", "DG_QLD1", "DG_SA1", "DG_TAS1", "KIDSPHG2"])),
        lit("Battery Storage")
    ).otherwise(col("Fuel Source - Primary"))
).withColumn(
    "Fuel Source - Descriptor",
    when(
        (col("DISPATCHTYPE") != "LOAD") &
        (col("STARTTYPE") != "NOT DISPATCHED") &
        (col("Fuel Source - Primary").isNull()) &
        (~col("DUID").isin(["DG_NSW1", "DG_VIC1", "DG_QLD1", "DG_SA1", "DG_TAS1", "KIDSPHG2"])),
        lit("Grid")
    ).otherwise(col("Fuel Source - Descriptor"))
).withColumn(
    "Fuel Source - Primary",
    when(
        (col("DUID").isin(["DG_NSW1", "DG_VIC1", "DG_QLD1", "DG_SA1", "DG_TAS1"])),
        lit("Aggregated")
    ).otherwise(col("Fuel Source - Primary"))
)

# Rename columns to replace hyphens with underscores for Delta Lake compatibility
nsw_dictionary_mapped = nsw_dictionary_mapped.withColumnRenamed("Fuel Source - Primary", "FUELSOURCEPRIMARY") \
    .withColumnRenamed("Fuel Source - Descriptor", "FUELSOURCEDESCRIPTOR")

# ========================================================================
# ENRICH WITH DUDETAIL CAPACITY DATA
# ========================================================================
print("\n" + "="*70)
print("Enriching with DUDETAIL capacity data")
print("="*70)

# Deduplicate nsw_dictionary_second by DUID, keeping highest VERSIONNO
window_spec = Window.partitionBy("DUID").orderBy(col("VERSIONNO").desc())

nsw_dictionary_second_latest = nsw_dictionary_second.withColumn(
    "row_num", 
    row_number().over(window_spec)
).filter(
    col("row_num") == 1
).drop("row_num")

print(f"\nℹ️  Original nsw_dictionary_second rows: {nsw_dictionary_second.count():,}")
print(f"ℹ️  Deduplicated rows (latest VERSIONNO per DUID): {nsw_dictionary_second_latest.count():,}")

# Select only the columns we need from nsw_dictionary_second
nsw_dict_second_cols = nsw_dictionary_second_latest.select(
    "DUID",
    "REGISTEREDCAPACITY",
    "AGCCAPABILITY", 
    "MAXCAPACITY",
    "NORMALLYONFLAG"
)

# Join with nsw_dictionary_mapped to add the new columns
nsw_dictionary_mapped = nsw_dictionary_mapped.join(
    nsw_dict_second_cols,
    on="DUID",
    how="left"
)

print(f"✅ Added columns: REGISTEREDCAPACITY, AGCCAPABILITY, MAXCAPACITY, NORMALLYONFLAG")

# Save nsw_dictionary_mapped to Delta Lake
nsw_dictionary_mapped.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("energy_endava_193.default.nsw_dictionary_mapped")
print("✅ nsw_dictionary_mapped saved to Delta Lake")

print(f"📊 Final row count: {nsw_dictionary_mapped.count():,}")
print("="*70)

display(nsw_dictionary_mapped)

In [0]:
# Display all qualified generators's "FUELSOURCEPRIMARY", including types of fuel and total counts
display(
    nsw_dictionary_mapped.filter((col("DISPATCHTYPE") != "LOAD") & (col("STARTTYPE") != "NOT DISPATCHED")).groupBy("FUELSOURCEPRIMARY").count().orderBy("count", ascending=False)
)

In [0]:
nsw_dispatch_demand = nsw_demand.join(
    nsw_dispatch_5min,
    on=["SETTLEMENTDATE", "REGIONID", "INTERVENTION"],
    how="inner"
)

# Save nsw_dispatch_demand_df to Delta Lake
nsw_dispatch_demand.write.mode("overwrite").saveAsTable("energy_endava_193.default.nsw_dispatch_demand")
print("✅ nsw_dispatch_demand saved to Delta Lake")

display(nsw_dispatch_demand)

In [0]:
# Find a specific day within November 2022 for a peak TOTALDEMAND
from pyspark.sql.functions import year, month, dayofmonth, max, col

# Filter for November 2024
nov_2022_df = nsw_dispatch_demand.filter(
    (year(col("SETTLEMENTDATE")) == 2022) &
    (month(col("SETTLEMENTDATE")) == 11)
)

# Find the day with the highest TOTALDEMAND peak
peak_day_df = nov_2022_df.groupBy(dayofmonth(col("SETTLEMENTDATE")).alias("day")) \
    .agg(max(col("TOTALDEMAND")).alias("peak_demand")) \
    .orderBy(col("peak_demand").desc()) \
    .limit(1)

# Get the day value
peak_day = peak_day_df.collect()[0]["day"]

# Create the dataframe for all times in the peak day
nsw_dispatch_demand_peak_2022 = nov_2022_df.filter(dayofmonth(col("SETTLEMENTDATE")) == peak_day)

# Save nsw_dispatch_demand_peak_2022 to Delta Lake
nsw_dispatch_demand_peak_2022.write.mode("overwrite").saveAsTable("energy_endava_193.default.nsw_dispatch_demand_peak_2022")
print("✅ nsw_dispatch_demand_peak_2022 saved to Delta Lake")

display(nsw_dispatch_demand_peak_2022)

In [0]:
from pyspark.sql.functions import to_date, col, lit

nsw_scada_peak_2022 = nsw_scada.filter(
    to_date(col("SETTLEMENTDATE")) == lit("2022-11-01")
)

# Save nsw_scada_peak_2022 to Delta Lake
nsw_scada_peak_2022.write.mode("overwrite").saveAsTable("energy_endava_193.default.nsw_scada_peak_2022")
print("✅ nsw_scada_peak_2022 saved to Delta Lake")

display(nsw_scada_peak_2022)

In [0]:
from pyspark.sql.functions import to_date, col, lit

# Filter for generators that were ACTIVE/OPERATIONAL on 2022-11-01
# (not generators that started on that exact date)
nsw_dictionary_peak_2022 = nsw_dictionary_mapped.filter(
    (to_date(col("START_DATE")) <= lit("2022-11-01")) &
    ((col("END_DATE").isNull()) | (to_date(col("END_DATE")) >= lit("2022-11-01")))
)

# Save nsw_dictionary_peak_2022 to Delta Lake
nsw_dictionary_peak_2022.write.mode("overwrite").saveAsTable("energy_endava_193.default.nsw_dictionary_peak_2022")
print("✅ nsw_dictionary_peak_2022 saved to Delta Lake")

display(nsw_dictionary_peak_2022)